# Checking for duplicates in database

This notebook analyses if there are any duplicates inside the tables of database.

In [51]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from database.scripts.config import load_config
from database.scripts.connect import connect

# ignore warings
import warnings
warnings.filterwarnings("ignore")

# plot defaults
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# load db config data
config = load_config("database/scripts/database.ini")
# connect to posgres db
conn = connect(config)

# execute sql and return results as DataFrame
def q(sql, params=None):
    return pd.read_sql(sql, conn, params=params)

Connected to the PostgreSQL server.


## Entries in work

In [52]:
q("""
    SELECT count(*)
    FROM openalex.work;
""")

,count
0,3860


## Duplicates in work

Checking for duplicates on entries in doi column.

In [53]:
work_doi = q("""
    SELECT doi, count(*) AS n, array_agg(id) AS ids
    FROM openalex.work
    WHERE doi IS NOT NULL
    GROUP BY doi
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{work_doi["n"].sum()} duplicates found")

work_doi

0 duplicates found


,doi,n,ids


## Handle duplicate DOIs in work table

When inserting works from OpenAlex data duplicates on DOI can appear. To address this issue the one with the newer publication year will be kept.

#### <u>Insert logic</u>

- If an existing work with same DOI is **newer or equally recent** -> skip the insert.
- If an existing work with same DOI is **older** -> delete it, then insert the new one.
- IF **no existing work** has this DOI -> insert it directly.


Investigations showed that whenever inserting new works a duplicate DOI can occur. The existing work in the database was always the **older** publication. Means, the publication year was less then or equal to the current one.

#### <u>Pseudocode within insertion process</u>
```python
existing_publication_year = get_work_publication_year(doi)
if existing_publication_year is not None:
    if existing_publication_year >= current_publication_year:
        return  # keep work with existing DOI
    delete_work(doi) # delete existing work
insert_work() # insert current work
```

Checking for duplicates on entries in normalized title column.

In [54]:
work_title = q("""
    SELECT lower(trim(title)) AS title_norm, count(*) AS n, array_agg(id) AS ids
    FROM openalex.work
    GROUP BY title_norm
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{work_title["n"].sum()} duplicates found")

work_title

0 duplicates found


,title_norm,n,ids


## Handle duplicate normalized titles in work table

When inserting works from OpenAlex data, duplicates on normalized title can appear even when DOIs differ. To resolve this issue, the exisiting work and the current work are compared field by field, in priority oder: `publication_date`,`authors_count`,`updated_date`. The first field where one side strictly wins decides the outcome; ties on a field fall thorugh to the next one.

#### <u>Insert logic</u>

- If the current work has **no publication_date** -> skip the insert (existing work is kept).
- Otherwise, compare `publication_date`, then `authors_count`, then `updated_date`, in order:
  - First field where **current > existing** -> delete the existing work, keep the current one.
  - First field where **current < existing** -> keep the existing work, skip the insert.
  - Equal on a field -> move to the next field in the list.
- If **all three fields are tied** -> keep the existing work, skip the insert.
- If **no existing work** has this normalized title -> insert it directly.

#### <u>Pseudocode within insertion process</u>
```python
if exists_normalized_title(title):
    existing_work = get_work_by_title(title)
    if current_publication_date is None:
        return # skip: current work misses publication_date
    for field in ["publication_date", "authors_count", "updated_date"]:
        if current[field] > existing[field]:
            delete_work(existing_work.id)
            break
        if current[field] < existing[field]
            return # keep existing work
    else:
        return # all fields tied -> keep existing work
insert_work() # no title duplicate or current work won on fields
```

## Entries in author

In [55]:
q("""
    SELECT count(*)
    FROM openalex.author;
""")

,count
0,14352


## Duplicates in author

In [56]:
author = q("""
    SELECT lower(trim(orcid)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.author
    GROUP BY orcid
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{author["n"].sum()} duplicates found")

author

6946 duplicates found


,lower,n,ids
0,NaN,6936,"[7987, 7988, 7738, 7727, 7992, 7993, 7994, 799..."
1,https://orcid.org/0000-0001-6825-4697,2,"[15113, 11525]"
2,https://orcid.org/0000-0002-6996-8859,2,"[13705, 13315]"
3,https://orcid.org/0000-0003-2036-0989,2,"[10153, 10151]"
4,https://orcid.org/0000-0003-4313-938x,2,"[13089, 597]"
5,https://orcid.org/0009-0007-6299-2008,2,"[5114, 5113]"


## Entries in institution

In [57]:
q("""
    SELECT count(*)
    FROM openalex.institution;
""")

,count
0,3731


## Duplicates in institution

In [58]:
institution = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.institution
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{institution["n"].sum()} duplicates found")

institution

49 duplicates found


,lower,n,ids
0,institut de recherche pour le développement,3,"[I4210108561, I1306264927, I4210166444]"
1,arab open university,2,"[I4210139873, I4210090878]"
2,bioinformatics institute,2,"[I4210148498, I4210137637]"
3,carter center,2,"[I1292524976, I4210096273]"
4,cisco systems (united states),2,"[I4210129566, I135428043]"
5,cracow university of technology,2,"[I24881138, I4210092770]"
6,institute of mechanics,2,"[I4210148896, I4210157653]"
7,institute of philosophy,2,"[I4210088627, I4210104258]"
8,institute of physics,2,"[I4210086947, I4210159876]"
9,joint research centre,2,"[I4210162697, I4210118689]"


## Entries in funder

In [59]:
q("""
    SELECT count(*)
    FROM openalex.funder;
""")

,count
0,935


## Duplicates in funder

In [60]:
funder = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.funder
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{funder["n"].sum()} duplicates found")

funder

2 duplicates found


,lower,n,ids
0,national science and technology council,2,"[F4320331164, F2461203286]"


## Entries in source

In [61]:
q("""
    SELECT count(*)
    FROM openalex.source;
""")

,count
0,2099


## Duplicates in source

In [62]:
source = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.source
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{source["n"].sum()} duplicates found")

source

27 duplicates found


,lower,n,ids
0,spire - sciences po institutional repository,5,"[S4406922454, S4406922461, S4406922398, S44069..."
1,geodesy and cartography,2,"[S12104227, S4210169005]"
2,infoscience (ecole polytechnique fédérale de l...,2,"[S4306400487, S4306400488]"
3,international journal of computer science and ...,2,"[S154051528, S4390963318]"
4,international journal of educational development,2,"[S5407050336, S20152851]"
5,journal of geophysical research atmospheres,2,"[S207178839, S4210205282]"
6,london school of economics and political scien...,2,"[S4306401594, S4306401593]"
7,pub – publications at bielefeld university (bi...,2,"[S4306401671, S4306401670]"
8,quaestiones geographicae,2,"[S157707066, S4210171875]"
9,repository@nottingham (university of nottingham),2,"[S4306402481, S4306402483]"


## Entries in locations

In [63]:
q("""
    SELECT count(*)
    FROM openalex.locations;
""")

,count
0,7489


## Duplicates in locations

In [64]:
locations = q("""
    SELECT lower(trim(pdf_url)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.locations
    GROUP BY pdf_url
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{locations["n"].sum()} duplicates found")

locations

4013 duplicates found


,lower,n,ids
0,None,4013,"[oai:mdpi.com:/2073-8994/15/5/1020/, 41646097,..."


## Entries in keyword

In [65]:
q("""
    SELECT count(*)
    FROM openalex.keyword;
""")

,count
0,5167


## Duplicates in keyword

In [66]:
keyword = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.keyword
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{keyword["n"].sum()} duplicates found")

keyword

0 duplicates found


,lower,n,ids


## Entries in domain

In [67]:
q("""
    SELECT count(*)
    FROM openalex.domain;
""")

,count
0,4


## Duplicates in domain

In [68]:
domain = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.domain
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{domain["n"].sum()} duplicates found")

domain

0 duplicates found


,lower,n,ids


## Entries in field

In [69]:
q("""
    SELECT count(*)
    FROM openalex.field;
""")

,count
0,26


## Duplicates in field

In [70]:
field = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.field
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{field["n"].sum()} duplicates found")

field

0 duplicates found


,lower,n,ids


## Entries in subfield

In [71]:
q("""
    SELECT count(*)
    FROM openalex.subfield;
""")

,count
0,202


## Duplicates in subfield

In [72]:
subfield = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.subfield
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{subfield["n"].sum()} duplicates found")

subfield

8 duplicates found


,lower,n,ids
0,industrial and manufacturing engineering,2,"[2209, 2311]"
1,genetics,2,"[1311, 2716]"
2,pharmacology,2,"[2736, 3004]"
3,neurology,2,"[2808, 2728]"


## Entries in topic

In [73]:
q("""
    SELECT count(*)
    FROM openalex.topic;
""")

,count
0,1613


## Duplicates in topic

In [74]:
topic = q("""
    SELECT lower(trim(display_name)), count(*) AS n, array_agg(id) AS ids
    FROM openalex.topic
    GROUP BY display_name
    HAVING count(*) > 1
    ORDER BY n DESC;
""")

print(f"{topic["n"].sum()} duplicates found")

topic

0 duplicates found


,lower,n,ids


In [75]:
conn.close()